In [9]:
!pip install -q gradio transformers torch black autopep8 reportlab requests
import gradio as gr
from transformers import pipeline
import torch
import ast
import re
from datetime import datetime
from pathlib import Path
import json
import black
import autopep8
from reportlab.lib.pagesizes import letter,A4
from reportlab.platypus import SimpleDocTemplate,Paragraph,Spacer,PageBreak,Table,TableStyle
from reportlab.lib.styles import getSampleStyleSheet,ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests

In [11]:
class ChatbotConfig:
  MODEL_NAME ="ibm-granite/granite-3,3-2b-instruct"
  DEFAULT_TEMP =0.7
  DEFAULT_MAX_TOKENS =256
  DEFAULT_TOP_P =0.95
conversation_history =[]
model_pipe =None
def load_model():
  global model_pipe
  print("Loading IBM Granite 3.3 2B Instruct model...")
  device_idx = 0 if torch.cuda.is_available() else -1
  device_type = "GPU" if device_idx == 0 else "CPU"
  print(f"  Using: {device_type}")
  model_pipe = pipeline(
      "text-generation",
      model=ChatbotConfig.MODEL_NAME,
      device=device_idx,
      torch_dtype=torch.float16 if device_idx == 0 else torch.float32,
  )
  print("Model loaded successfully!")
  return model_pipe

In [13]:
def detect_python_code(text):
   code_patterns =[r'def\s+\w+\s*\(',r'class\s+\w+',r'import\s+\w+',r'from\s+\w+\s+import',r'for\s+.*\s+in\s+',r'while\s+.*:']
   return any (re.search(pattern,text)for pattern in code_patterns)

In [2]:
def check_syntax(code):
  try:
    ast.parse(code)
    return True, None
  except SyntaxError as e:
    return False, f"Syntax Error (line {e.lineno}): {e.msg}"
  except Exception as e:
    return False, f"Error: {str(e)}"

In [4]:
def fix_code_indentation(code):
  lines=code.split('\n')
  fixed_lines=[line.rstrip().replace('\t','  ') for line in lines]
  return '\n'.join(fixed_lines)

def fix_code_brackets(code):
  # Balance parentheses
  paren_balance = code.count('(') - code.count(')')
  if paren_balance > 0:
      code += ')' * paren_balance

  # Balance curly braces
  curly_balance = code.count('{') - code.count('}')
  if curly_balance > 0:
      code += '}' * curly_balance

  # Balance square brackets
  square_balance = code.count('[') - code.count(']')
  if square_balance > 0:
      code += ']' * square_balance

  return code

def format_code(code):
  try:
      return black.format_str(code, mode=black.FileMode())
  except Exception:
      try:
          return autopep8.fix_code(code)
      except Exception:
          return code

In [6]:
def process_user_input(user_message,temperature,max_tokens,top_p,chat_history):
  if not user_message.strip():
    return chat_history

  response_text = ""
  ai_prompt = ""

  if detect_python_code(user_message):
    code_pattern = r"```(?:python\n)?(.*?)```"
    code_match = re.search(code_pattern, user_message, re.DOTALL)

    if code_match:
      code_snippet = code_match.group(1).strip()

      fixed_code = fix_code_indentation(code_snippet)
      fixed_code = fix_code_brackets(fixed_code)
      fixed_code = format_code(fixed_code)

      is_valid, syntax_error_msg = check_syntax(fixed_code)

      if is_valid:
        response_text = f"I've identified and fixed your Python code.\n\n**Fixed Code:**\n```python\n{fixed_code}\n```"
        ai_prompt = f"I fixed this python code:\n```python\n{fixed_code}\n```\nBriefly explain what it does."
      else:
        response_text = f"I've attempted to fix your Python code, but it still contains syntax errors.\n\n**Original Code:**\n```python\n{code_snippet}\n```\n\n**Attempted Fix:**\n```python\n{fixed_code}\n```\n\n**Error:** {syntax_error_msg}"
        ai_prompt = f"This python code still has errors after an attempted fix:\n```python\n{fixed_code}\n```\nIt has the error: {syntax_error_msg}. Briefly explain the error and suggest how to fix it."
    else:
      response_text = "I detected what might be Python code, but I couldn't find a properly formatted code block (e.g., enclosed in triple backticks ```). Please ensure your code is correctly formatted."
      ai_prompt = f"The user provided input that seems like code but is not well-formatted. User input: '{user_message}'. Explain that code blocks need triple backticks."
  else:
    response_text = "Thinking..."
    ai_prompt = user_message

  ai_response = ""
  if ai_prompt and model_pipe:
    try:
      outputs = model_pipe(
          ai_prompt,
          max_new_tokens=max_tokens,
          temperature=temperature,
          top_p=top_p,
          do_sample=True,
          num_return_sequences=1
      )
      ai_response = outputs[0]['generated_text']
      if ai_response.startswith(ai_prompt):
          ai_response = ai_response[len(ai_prompt):].strip()
    except Exception as e:
      ai_response = f"Error generating AI response: {str(e)}"
      print(f"Error calling model_pipe: {e}")
  elif not model_pipe:
      ai_response = "Error: Model not loaded. Please run `load_model()` first."

  chat_history.append((user_message, response_text + "\n\nAI: " + ai_response))
  return chat_history

In [ ]:
def generate_response(user_message,temperature,max_tokens,top_p):
  conversation_history.append({"rele":"user","content": user_message})
  try:
    response=modle_pipe(conversation_history,max_new_tokens=int(max_tokens),tempeature=tempeature,do_sample=true,top_p=top)
    assistant_message=response[0]['generated_text']
    if ifinstant(assistant_message,list):
        Assisstant_message=assistant_message[-1].get("content",str(assistant_messagg))
    conversation_history.append({"role":"assistant","content":assistant_message})
    return assistant_message
 except exception as e:
    error_response=f" error:{str(e)}"
    conversation_history>append({"role:"assistant","content:erroe_responce})
    return error_response





